# Example of Pyspark ML

In [99]:
import findspark
findspark.init()

In [100]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Part5').getOrCreate()

In [101]:
spark

In [102]:
training = spark.read.csv('test.csv', header=True, inferSchema=True)

In [103]:
type(training)

pyspark.sql.dataframe.DataFrame

In [104]:
training = training.na.drop()

In [105]:
training.show()

+-------+---+----------+------+
|   Name|Age|Experience|Salary|
+-------+---+----------+------+
|  Surya| 25|        10| 30000|
|Lakshmi| 22|         8| 25000|
|  Nanna| 50|         4| 20000|
|   Amma| 45|        15| 15000|
| Sanjay| 21|         1| 18000|
|  Sahoo| 23|         2| 40000|
| Mahesh| 25|         3| 38000|
+-------+---+----------+------+



In [106]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [107]:
training.columns

['Name', 'Age', 'Experience', 'Salary']

In [108]:
# [Age, Experience] -- new feature --> independent feature

In [109]:
from pyspark.ml.feature import VectorAssembler
featrueassembler = VectorAssembler(inputCols=["Age", "Experience"], outputCol="Independent_Features")

In [110]:
output = featrueassembler.transform(training)

In [111]:
output.columns

['Name', 'Age', 'Experience', 'Salary', 'Independent_Features']

In [112]:
type(output)

pyspark.sql.dataframe.DataFrame

In [116]:
output.show()

+-------+---+----------+------+--------------------+
|   Name|Age|Experience|Salary|Independent_Features|
+-------+---+----------+------+--------------------+
|  Surya| 25|        10| 30000|         [25.0,10.0]|
|Lakshmi| 22|         8| 25000|          [22.0,8.0]|
|  Nanna| 50|         4| 20000|          [50.0,4.0]|
|   Amma| 45|        15| 15000|         [45.0,15.0]|
| Sanjay| 21|         1| 18000|          [21.0,1.0]|
|  Sahoo| 23|         2| 40000|          [23.0,2.0]|
| Mahesh| 25|         3| 38000|          [25.0,3.0]|
+-------+---+----------+------+--------------------+



In [114]:
finalized_data = output.select("Independent_Features", "Salary")

In [115]:
finalized_data.show()

+--------------------+------+
|Independent_Features|Salary|
+--------------------+------+
|         [25.0,10.0]| 30000|
|          [22.0,8.0]| 25000|
|          [50.0,4.0]| 20000|
|         [45.0,15.0]| 15000|
|          [21.0,1.0]| 18000|
|          [23.0,2.0]| 40000|
|          [25.0,3.0]| 38000|
+--------------------+------+



In [117]:
from pyspark.ml.regression import LinearRegression
train_data, test_data = finalized_data.randomSplit([0.75, 0.25])
regressor = LinearRegression(featuresCol='Independent_Features',labelCol='Salary')
regressor = regressor.fit(train_data)

In [118]:
# Coefficients
regressor.coefficients

DenseVector([-300.724, -126.392])

In [119]:
# Intercepts
regressor.intercept

37409.86036890191

In [120]:
# Predictions
pred_results = regressor.evaluate(test_data)

In [121]:
pred_results.predictions.show()

+--------------------+------+------------------+
|Independent_Features|Salary|        prediction|
+--------------------+------+------------------+
|         [45.0,15.0]| 15000|21981.399758662323|
+--------------------+------+------------------+



In [122]:
pred_results.meanAbsoluteError, pred_results.meanSquaredError

(6981.399758662323, 48739942.590250336)